# Liveness 06 · Export (ONNX · LiteRT · Core ML)

Device: both MiniFASNet scales (2.7 and 4.0) → `.tflite` / `.mlpackage`. Server: ViT-S + merged LoRA → ONNX.
Package `Weights/liveness/<version>/` with `Model_Card.json` incl. the default threshold.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Liveness
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import torch
from Common_Utils.Export_Utils import export_onnx, export_litert, export_coreml, write_model_card
VERSION = "1.0.0"
OUT = model_dir("liveness", VERSION)
sample = torch.zeros(1, 3, 80, 80)

class Logits_Only(torch.nn.Module):          # export only the classification output (Fourier head is training-only)
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, x): return torch.softmax(self.net(x)[0], dim=1)[:, 1:2]

# for scale in (2.7, 4.0):
#     net = MiniFASNetV2SE(); net.load_state_dict(torch.load(RUNS / f"minifasnet_s{scale}.pt", map_location="cpu"))
#     export_litert(Logits_Only(net), sample, OUT / f"liveness_s{scale}.tflite", int8=True)
#     export_coreml(Logits_Only(net), sample, OUT / f"liveness_s{scale}.mlpackage")
# server: model = model.merge_and_unload(); export_onnx(model, torch.zeros(1, 3, 224, 224), OUT / "liveness_server_vit.onnx")

In [ ]:
write_model_card(OUT, kind="liveness", version=VERSION,
    metrics={"apcer": None, "bpcer": None, "acer": None, "cross_dataset_acer": None, "default_threshold": None},
    data_sources=["CelebA-Spoof", "OULU-NPU", "SiW / SiW-Mv2", "In-house consented attacks"],
    input_spec={"device": "1x3x80x80 RGB 0-1, face box x2.7 and x4.0", "server": "1x3x224x224 timm DINOv2 normalisation"},
    notes="Thresholds are tuned per station (lighting / camera) and stored in tenant_ai_models.config.")